In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 50)

DATA = Path("../Data/Original_Datasets")

In [2]:
files = {
    "customers": "SwiftLog_Customers.csv",
    "orders": "SwiftLog_Orders.csv",
    "order_items": "SwiftLog_Order_Items.csv",
    "products": "SwiftLog_Products.csv",
    "suppliers": "SwiftLog_Suppliers.csv",
    "warehouses": "SwiftLog_Warehouses.csv",
    "inventory": "SwiftLog_Inventory.csv",
    "shipments": "SwiftLog_Shipments.csv",
    "delivery": "SwiftLog_Delivery_Performance.csv",
}
dfs = {name: pd.read_csv(DATA / f) for name, f in files.items()}

for name, df in dfs.items():
    print(f"{name:12s} shape={df.shape}  nulls={df.isna().sum().sum()}  dups={df.duplicated().sum()}")

customers    shape=(1000, 3)  nulls=0  dups=0
orders       shape=(5000, 4)  nulls=0  dups=0
order_items  shape=(15000, 4)  nulls=0  dups=0
products     shape=(250, 3)  nulls=0  dups=0
suppliers    shape=(150, 2)  nulls=0  dups=0
warehouses   shape=(15, 2)  nulls=0  dups=0
inventory    shape=(4000, 3)  nulls=0  dups=3
shipments    shape=(5000, 4)  nulls=0  dups=0
delivery     shape=(5000, 3)  nulls=0  dups=0


In [3]:
dfs["orders"]["OrderDate"] = pd.to_datetime(dfs["orders"]["OrderDate"], format="%m/%d/%Y")
print(dfs["orders"]["OrderDate"].min(), dfs["orders"]["OrderDate"].max())

o, oi, s, d, p, inv, w, c = (dfs[k] for k in
    ["orders","order_items","shipments","delivery","products","inventory","warehouses","customers"])

print("orders -> customers:", o.CustomerID.isin(c.CustomerID).mean())
print("items -> orders:", oi.OrderID.isin(o.OrderID).mean())
print("items -> products:", oi.ProductID.isin(p.ProductID).mean())
print("shipments -> orders:", s.OrderID.isin(o.OrderID).mean())
print("shipments -> warehouses:", s.WarehouseID.isin(w.WarehouseID).mean())
print("delivery -> shipments:", d.ShipmentID.isin(s.ShipmentID).mean())
print("inventory -> products:", inv.ProductID.isin(p.ProductID).mean())

2026-01-01 00:00:00 2026-09-08 00:00:00
orders -> customers: 1.0
items -> orders: 1.0
items -> products: 1.0
shipments -> orders: 1.0
shipments -> warehouses: 1.0
delivery -> shipments: 1.0
inventory -> products: 1.0


Data Cleaning

In [6]:
# Inventory duplicates 
inv = dfs["inventory"]
print("exact dup rows:", inv.duplicated().sum())
print("repeated WarehouseID+ProductID pairs:", inv.duplicated(["WarehouseID","ProductID"]).sum())
print(inv[inv.duplicated(["WarehouseID","ProductID"], keep=False)].sort_values(["WarehouseID","ProductID"]).head(8))

# Cancelled orders: do they still have shipments and delays
m = o.merge(s, on="OrderID").merge(d[["ShipmentID","Status"]], on="ShipmentID", suffixes=("_order","_delivery"))
print(pd.crosstab(m["Status_order"], m["Status_delivery"], normalize="index").round(3))

# Orders with no items, and September volume
print("orders with no items:", (~o.OrderID.isin(oi.OrderID)).sum())
print(o.groupby(o.OrderDate.dt.to_period("M")).size())

# Value ranges
print(oi.Quantity.describe()[["min","max"]].to_dict(), p.UnitPrice.describe()[["min","max"]].to_dict())
print("stock zero:", (inv.Stock == 0).sum(), " max:", inv.Stock.max())

exact dup rows: 3
repeated WarehouseID+ProductID pairs: 1545
     WarehouseID ProductID  Stock
344          W01      P001    154
500          W01      P001    127
3140         W01      P001    456
1470         W01      P007     93
2458         W01      P007    319
3268         W01      P007    147
979          W01      P011    182
3293         W01      P011    317
Status_delivery  Delayed  On Time
Status_order                     
Cancelled          0.645    0.355
Delivered          0.628    0.372
Pending            0.639    0.361
orders with no items: 236
OrderDate
2026-01    594
2026-02    577
2026-03    597
2026-04    638
2026-05    612
2026-06    579
2026-07    595
2026-08    654
2026-09    154
Freq: M, dtype: int64
{'min': 1.0, 'max': 10.0} {'min': 11.78, 'max': 496.61}
stock zero: 9  max: 500


Processed Datesets


In [7]:
OUT = Path("../Data/Processed_Datasets")
OUT.mkdir(parents=True, exist_ok=True)

# Inventory: drop dups, then one row per warehouse + product
inv_clean = (inv.drop_duplicates()
                .groupby(["WarehouseID","ProductID"], as_index=False)
                .agg(Stock=("Stock","sum"), Stock_Records=("Stock","size")))

# Order lines: items + product info + order info (cancelled orders removed)
lines = (oi.merge(p, on="ProductID", how="left", validate="many_to_one")
           .merge(o, on="OrderID", how="left", validate="many_to_one"))
lines["LineValue"] = lines["Quantity"] * lines["UnitPrice"]
lines = lines.merge(c[["CustomerID","City"]], on="CustomerID", how="left", validate="many_to_one")
lines_clean = lines[lines["Status"] != "Cancelled"].copy()

# Shipments: shipment + delivery status + order + warehouse (all orders kept)
ship = (s.merge(d[["ShipmentID","Status"]].rename(columns={"Status":"DeliveryStatus"}),
                on="ShipmentID", how="left", validate="one_to_one")
          .merge(o, on="OrderID", how="left", validate="one_to_one")
          .merge(w, on="WarehouseID", how="left", validate="many_to_one")
          .merge(c[["CustomerID","City"]], on="CustomerID", how="left", validate="many_to_one")
          .rename(columns={"Location":"WarehouseCity","City":"CustomerCity","Status":"OrderStatus"}))

# Sanity checks
assert len(inv_clean) == 2455
assert len(lines) == 15000 and lines["UnitPrice"].notna().all()
assert len(ship) == 5000 and ship["DeliveryStatus"].notna().all()
assert ((ship.DelayDays >= 3) == (ship.DeliveryStatus == "Delayed")).all()

inv_clean.to_csv(OUT / "inventory_clean.csv", index=False)
lines_clean.to_csv(OUT / "order_lines_clean.csv", index=False)
ship.to_csv(OUT / "shipments_full.csv", index=False)
print(inv_clean.shape, lines_clean.shape, ship.shape)

(2455, 4) (10074, 11) (5000, 10)


In [8]:
lines.to_csv(OUT / "order_lines_all.csv", index=False)
print(lines.shape)   

(15000, 11)
